# Rock in the stream

**What you will learn:** when to use `avoid` for a hard constraint versus a
running-cost penalty, and why the two answers differ numerically.

**pyspect API:** `TVHJImpl.avoid` (pyspect) vs `hj.solve` with
`hamiltonian_postprocessor` (raw hj_reachability)

A canoe drifts downstream and steers sideways only. From which positions is
hitting the rock unavoidable?


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from matplotlib.patches import FancyArrowPatch

import hj_reachability as hj
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import DriftingCanoe

In [ ]:
T = 20              # horizon, in seconds
DOMAIN = [-10, 10]

def build(vox):
    """Grid of vox+1 points per side and 2*vox+1 time steps, as in hjr_examples."""
    axes = [dict(name='t', bounds=[0, T],   points=2 * vox + 1),
            dict(name='x', bounds=DOMAIN,   points=vox + 1),
            dict(name='y', bounds=DOMAIN,   points=vox + 1)]
    # Only the final value function is needed here, so skip stacking the time axis
    impl = TVHJImpl(dict(cls=DriftingCanoe), axes, accuracy='very_high', _stack=False)
    S = impl.grid.states
    rock = jnp.sqrt(S[..., 0]**2 + (S[..., 1] + 3.0)**2) - 2.0
    return impl, rock

In [ ]:
# The setting: drift downwards, steering sideways, one rock
fig, ax = plt.subplots(figsize=(5.5, 5.5))
th = np.linspace(0, 2 * np.pi, 100)
ax.fill(2 * np.cos(th), -3 + 2 * np.sin(th), color='dimgray', label='Rock')
gx, gy = np.meshgrid(np.linspace(-8, 8, 9), np.linspace(-8, 8, 9))
ax.quiver(gx, gy, 0.0 * gx, -1.0 + 0.0 * gy, color='b', alpha=0.5, label='Current')
ax.set_xlim(DOMAIN); ax.set_ylim(DOMAIN); ax.set_aspect('equal')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('The canoe drifts down and can steer in $x$')
ax.legend(loc='upper right', framealpha=1.0)
plt.show()

## Approach 1 - a penalty in the running cost

The penalty is added to the Hamiltonian, which is not a set operation, so it has no
`pyspect` equivalent by design. It is written here with the raw `hj_reachability` API,
reusing the dynamics object that `TVHJImpl` already built.

The value function starts at zero everywhere and accumulates `-10**exp` per unit of time
spent inside the rock, so `V < 0` means "a collision could not be avoided".


In [ ]:
def penalty_value(impl, rock, vox, exp):
    r = jnp.where(rock < 0.0, -(10.0**exp), 0.0)
    settings = hj.SolverSettings.with_accuracy(
        'very_high', hamiltonian_postprocessor=lambda H: H + jnp.minimum(r, 0.0))
    times = np.linspace(0.0, -T, 2 * vox + 1)
    V = hj.solve(settings, impl.avoid_dynamics, impl.grid, times, 0.0 * r,
                 progress_bar=False)
    return np.array(V[-1])

## Approach 2 - the avoid set

Here the rock is described by its signed distance, and `avoid` returns the states from
which the canoe cannot keep away from it. Nothing else to tune.


In [ ]:
VOXES = [500, 100, 20]
EXPS = [2, 4, 6]

setups = {v: build(v) for v in VOXES}
grids = {v: impl.grid for v, (impl, _) in setups.items()}

penalty, avoid = {}, {}
for v in VOXES:
    impl, rock = setups[v]
    for e in EXPS:
        penalty[(v, e)] = penalty_value(impl, rock, v, e)
    avoid[v] = np.array(impl.avoid(rock))[0]
    print(f'{v}x{v} done')

In [ ]:
def plot_value_function(V, grid, fig, ax, show_colorbar=True):
    V = np.clip(np.asarray(V), -10.0, +10.0)
    x = np.asarray(grid.states[:, 0, 0])
    y = np.asarray(grid.states[0, :, 1])

    mesh = ax.pcolormesh(x, y, V.T, cmap='viridis', vmin=-10.0, vmax=10.0,
                         shading='auto')
    # The unsafe set is the zero sublevel set; without this it is invisible
    ax.contour(x, y, V.T, levels=[0], colors='red', linewidths=1.6)
    th = np.linspace(0, 2 * np.pi, 100)
    ax.plot(2 * np.cos(th), -3 + 2 * np.sin(th), color='white', lw=1.5)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_aspect('equal')
    if show_colorbar:
        fig.colorbar(mesh, ax=ax, label='Value', fraction=0.035, pad=0.04)
    return mesh


def label_grid(fig, axs, row_labels, row_title, col_labels, col_title):
    """Row and column captions with the two arrows of the original slide."""
    fig.tight_layout(rect=[0.10, 0.02, 0.98, 0.88])

    top = max(ax.get_position().y1 for ax in axs.flat)
    label_y, arrow_y, title_y = top + 0.025, top + 0.065, top + 0.095

    for i, lab in enumerate(row_labels):
        pos = axs[i, 0].get_position()
        fig.text(0.06, (pos.y0 + pos.y1) / 2, lab, ha='center', va='center',
                 fontsize=15, fontweight='bold', rotation=90)
    for j, lab in enumerate(col_labels):
        pos = axs[0, j].get_position()
        fig.text((pos.x0 + pos.x1) / 2, label_y, lab, ha='center', va='center',
                 fontsize=15, fontweight='bold')

    lp, rp = axs[0, 0].get_position(), axs[0, -1].get_position()
    fig.add_artist(FancyArrowPatch(posA=(lp.x0, arrow_y), posB=(rp.x1, arrow_y),
                                   transform=fig.transFigure, arrowstyle='->',
                                   mutation_scale=22, lw=2, color='black'))
    fig.text((lp.x0 + rp.x1) / 2, title_y, col_title, ha='center', va='center',
             fontsize=16, fontweight='bold')

    if row_title:
        tp, bp = axs[0, 0].get_position(), axs[-1, 0].get_position()
        fig.add_artist(FancyArrowPatch(posA=(0.022, tp.y1), posB=(0.022, bp.y0),
                                       transform=fig.transFigure, arrowstyle='->',
                                       mutation_scale=22, lw=2, color='black'))
        fig.text(0.008, (tp.y1 + bp.y0) / 2, row_title, ha='center', va='center',
                 fontsize=16, fontweight='bold', rotation=90)

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(14, 12))
for i, e in enumerate(EXPS):
    for j, v in enumerate(VOXES):
        plot_value_function(penalty[(v, e)], grids[v], fig, axs[i, j])

label_grid(fig, axs,
           [f'Penalty $=10^{{{e}}}$' for e in EXPS], 'Increasing penalty',
           [f'${v}\\times{v}$' for v in VOXES], 'Decreasing resolution')
plt.show()

The red curve is the zero level, that is the boundary of "a collision happened". The
speckles all over the left column are not a plotting glitch, they are the heart of the
problem. Away from the rock the true value is exactly zero, so the value function is
**flat**, and the sign of a flat function is decided by rounding error. The zero level of
a running-cost value function is therefore not a well-defined object, whatever the
resolution.

Reading the panels left to right, the second effect appears: as the grid coarsens the
penalty diffuses upstream and paints a tall column above the rock as unsafe. At `20x20`
with a penalty of `10**6` the column reaches the top of the domain, which would tell the
canoe that a position ten units upstream is already lost.


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.6), squeeze=False)
for j, v in enumerate(VOXES):
    plot_value_function(avoid[v], grids[v], fig, axs[0, j])

label_grid(fig, axs, [''], '', [f'${v}\\times{v}$' for v in VOXES],
           'Decreasing resolution')
plt.show()

## Putting numbers on it

Both formulations are asked the same question: what fraction of the domain is unsafe? The
rock alone covers 3.1 %, and the true answer is only a little more, since a canoe starting
well above the rock has room to steer around it.


In [ ]:
def area(W):
    return 100 * float((np.asarray(W) < 0).mean())

print('unsafe fraction of the domain')
print(' ' * 16 + ''.join(f'{v}x{v}'.rjust(12) for v in VOXES))
for e in EXPS:
    print(f'  penalty 1e{e}'.ljust(16)
          + ''.join(f'{area(penalty[(v, e)]):10.1f} %' for v in VOXES))
print('  avoid'.ljust(16) + ''.join(f'{area(avoid[v]):10.1f} %' for v in VOXES))
print(f'\nrock alone: {100 * np.pi * 4 / 400:.1f} %')

And here is the flatness that makes the zero level unusable. The fraction below is how
much of the domain carries a value indistinguishable from zero, which is the region where
the sign of `V`, and therefore the answer, is decided by rounding error.


In [ ]:
def flat(W):
    return 100 * float((np.abs(np.asarray(W)) < 1e-3).mean())

print('fraction of the domain where |V| < 1e-3')
print(' ' * 16 + ''.join(f'{v}x{v}'.rjust(12) for v in VOXES))
for e in EXPS:
    print(f'  penalty 1e{e}'.ljust(16)
          + ''.join(f'{flat(penalty[(v, e)]):10.1f} %' for v in VOXES))
print('  avoid'.ljust(16) + ''.join(f'{flat(avoid[v]):10.1f} %' for v in VOXES))

## What this example is for

The avoid set sits at 3.3 % at both usable resolutions, against a true value just above the
3.1 % of the rock itself. It only loses ground at `20x20`, where a grid step of one unit
can no longer represent a disc of radius 2, and it errs on the small side there rather than
inventing danger.

The penalty answer is wrong in three separate ways. It already overstates the unsafe area
by a factor of two at `500x500`. It degrades to 16.6 %, a factor of five, once the grid is
coarse. And the magnitude of the penalty, a number nobody can derive from the physics,
changes the result on its own.

Underneath all three sits the flatness table. A running cost is zero wherever nothing
happens, so its value function is flat across most of the domain and its zero level set is
not a geometric object at all. A reachability value function crosses zero transversally,
which is exactly why the level set can be tracked reliably even on a coarse grid.

None of this argues against running costs in general, they are the right tool when you
really do want to trade off accumulated quantities. It argues against encoding a hard
constraint as a soft one. `avoid` asks the question that was actually being asked, and the
set it returns is the answer rather than a by-product.
